# N-gram Language Models

In this notebook we will work with different aspects of n-gram language models, including training, smoothing, text generation and evaluation. Additionally, we will empirically verify the Zipf's Law.

## Setup

In [1]:
!pip install matplotlib
!pip install nltk

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\User\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\User\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import nltk

nltk.download('gutenberg')
nltk.download('brown')
nltk.download('punkt')
nltk.download('punkt_tab')

import random
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter, defaultdict

from nltk.corpus import gutenberg, brown
from nltk.lm import MLE, Laplace, KneserNeyInterpolated
from nltk.lm.preprocessing import padded_everygram_pipeline, pad_both_ends, everygrams
from nltk.util import ngrams

random.seed(42)
np.random.seed(42)

[nltk_data] Downloading package gutenberg to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\gutenberg.zip.
[nltk_data] Downloading package brown to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\brown.zip.
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


## Warm-up: Building N-grams by Hand

Before using NLTK's high-level API, let's build bigrams and compute MLE probabilities manually. This ensures that the library is not a black box.

### Extracting bigrams with `zip`

In [3]:
sentence = "the cat sat on the mat and the cat wore a hat".split()

# A bigram is just every consecutive pair of tokens
bigrams_manual = list(zip(sentence[:-1], sentence[1:]))
print('Bigrams:', bigrams_manual)

Bigrams: [('the', 'cat'), ('cat', 'sat'), ('sat', 'on'), ('on', 'the'), ('the', 'mat'), ('mat', 'and'), ('and', 'the'), ('the', 'cat'), ('cat', 'wore'), ('wore', 'a'), ('a', 'hat')]


In [4]:
# Count unigrams and bigrams
unigram_counts = Counter(sentence)
bigram_counts  = Counter(bigrams_manual)

print('Unigram counts:', dict(unigram_counts))
print('Bigram counts: ', dict(bigram_counts))

Unigram counts: {'the': 3, 'cat': 2, 'sat': 1, 'on': 1, 'mat': 1, 'and': 1, 'wore': 1, 'a': 1, 'hat': 1}
Bigram counts:  {('the', 'cat'): 2, ('cat', 'sat'): 1, ('sat', 'on'): 1, ('on', 'the'): 1, ('the', 'mat'): 1, ('mat', 'and'): 1, ('and', 'the'): 1, ('cat', 'wore'): 1, ('wore', 'a'): 1, ('a', 'hat'): 1}


In [5]:
def mle_bigram_prob(w1, w2, unigram_counts, bigram_counts):
    """P(w2 | w1) = C(w1, w2) / C(w1)"""
    return bigram_counts[(w1, w2)] / unigram_counts[w1]

# Try some examples
print(f"P(cat | the)  = {mle_bigram_prob('the', 'cat', unigram_counts, bigram_counts):.3f}")
print(f"P(mat | the)  = {mle_bigram_prob('the', 'mat', unigram_counts, bigram_counts):.3f}")
print(f"P(sat | the)  = {mle_bigram_prob('the', 'sat', unigram_counts, bigram_counts):.3f}")

# What about an unseen bigram?
print(f"P(dog | the)  = {mle_bigram_prob('the', 'dog', unigram_counts, bigram_counts):.3f}")

P(cat | the)  = 0.667
P(mat | the)  = 0.333
P(sat | the)  = 0.000
P(dog | the)  = 0.000


## 2. NLTK Pipeline: Tokenisation and Padding

NLTK's `padded_everygram_pipeline` handles padding with `<s>` and `</s>` tokens automatically. Let's see what it produces.

For the experiments, we will work with  Austen's *Emma* corpus.

*Emma* (1815) is a novel by Jane Austen, included in NLTK's `gutenberg` collection, 
a set of public domain literary texts. It contains ~180,000 tokens of homogeneous 
19th-century narrative prose, which makes it a convenient and stylistically recognizable 
corpus for training and generating text.

In [6]:
# We will use Austen's Emma from project Gutenberg: rich vocabulary, manageable size
raw_tokens = gutenberg.words('austen-emma.txt')
sentences  = gutenberg.sents('austen-emma.txt')

# Lowercase everything
sentences = [[w.lower() for w in sent] for sent in sentences]

print(f'Total sentences : {len(sentences)}')
print(f'Total tokens    : {sum(len(s) for s in sentences)}')
print(f'Sample sentence : {sentences[10]}')

Total sentences : 7752
Total tokens    : 192484
Sample sentence : ['the', 'danger', ',', 'however', ',', 'was', 'at', 'present', 'so', 'unperceived', ',', 'that', 'they', 'did', 'not', 'by', 'any', 'means', 'rank', 'as', 'misfortunes', 'with', 'her', '.']


In [7]:
# Train / test split (90% / 10%)
split = int(len(sentences) * 0.9)
train_sents = sentences[:split]
test_sents  = sentences[split:]

print(f'Training sentences : {len(train_sents)}')
print(f'Test sentences     : {len(test_sents)}')

Training sentences : 6976
Test sentences     : 776


In [8]:
# Inspect what padded_everygram_pipeline produces for order n=2
n = 2
sample = [['the', 'cat', 'sat']]
train_data, vocab = padded_everygram_pipeline(n, sample)

print('All n-grams produced for ["the", "cat", "sat"] with n=2:')
for gram in list(padded_everygram_pipeline(n, sample)[0]):
    print(' ', list(gram))

All n-grams produced for ["the", "cat", "sat"] with n=2:
  [('<s>',), ('<s>', 'the'), ('the',), ('the', 'cat'), ('cat',), ('cat', 'sat'), ('sat',), ('sat', '</s>'), ('</s>',)]


Notice the `<s>` and `</s>` padding tokens. They allow the model to learn probabilities for the first and last word of a sentence.

## Training MLE, Laplace, and Kneser-Ney Models

In [9]:
n = 3  # trigram model

def train_model(model_class, n, sentences, **kwargs):
    train_data, vocab = padded_everygram_pipeline(n, sentences)
    model = model_class(n, **kwargs)
    model.fit(train_data, vocab)
    return model

model_mle = train_model(MLE, n, train_sents)
model_lap = train_model(Laplace, n, train_sents)
model_kn  = train_model(KneserNeyInterpolated, n, train_sents)

print(f'Vocabulary size: {len(model_mle.vocab)}')

Vocabulary size: 6917


In [10]:
# Compare probabilities for a seen trigram
context = ('she', 'was')
word    = 'very'

print(f"P('{word}' | '{context}')")
print(f"  MLE       : {model_mle.score(word, context):.6f}")
print(f"  Laplace   : {model_lap.score(word, context):.6f}")
print(f"  Kneser-Ney: {model_kn.score(word, context):.6f}")

P('very' | '('she', 'was')')
  MLE       : 0.043011
  Laplace   : 0.001807
  Kneser-Ney: 0.043581


In [11]:
# Now try an UNSEEN trigram
context = ('a', 'fantastic')
word    = 'person'

print(f"P('{word}' | '{context}')")
print(f"  MLE       : {model_mle.score(word, context):.6f}")
print(f"  Laplace   : {model_lap.score(word, context):.6f}")
print(f"  Kneser-Ney: {model_kn.score(word, context):.6f}")

P('person' | '('a', 'fantastic')')
  MLE       : 0.000000
  Laplace   : 0.000145
  Kneser-Ney: 0.000410


## 4. Text Generation

In [12]:
def generate_text(model, n_words=30, random_seed=123):
    """Generate text from a trained NLTK language model."""
    tokens = model.generate(n_words, random_seed=random_seed)
    # Filter out padding tokens
    tokens = [t for t in tokens if t not in ('<s>', '</s>')]
    return ' '.join(tokens)

print('=== MLE trigram ===')
print(generate_text(model_mle))

print('\n=== Laplace trigram ===')
print(generate_text(model_lap))

print('\n=== Kneser-Ney trigram ===')
print(generate_text(model_kn))

=== MLE trigram ===
, and it had not a little gruel to you ,"-- said emma , i am still afraid that it wins upon one directly .

=== Laplace trigram ===
, and it did not ; i happened to be engaged by no housekeeper ' s loss had been lately used to a soul that i may deserve him ;

=== Kneser-Ney trigram ===
, and it had not a little gruel to you ,"-- said emma , i am still afraid that it wins upon one directly .


In [13]:
# Compare bigram vs. trigram generation with MLE
model_mle_bi = train_model(MLE, 2, train_sents)

print('=== MLE bigram ===')
print(generate_text(model_mle_bi))

print('\n=== MLE trigram ===')
print(generate_text(model_mle))

=== MLE bigram ===
, and if he was a minute , to be easier to come !" -- in a great difference plain , and the agreeable the other ." seeing

=== MLE trigram ===
, and it had not a little gruel to you ,"-- said emma , i am still afraid that it wins upon one directly .


Higher-order models tend to produce more locally coherent text, but are more prone to copying the training data verbatim.

## Evaluation with Perplexity

In [14]:
def compute_perplexity(model, n, sentences):
    """
    Compute average perplexity over a list of sentences.
    Sentences containing OOV tokens are handled gracefully.
    """
    perplexities = []
    for sent in sentences:
        # Pad sentence and extract n-grams
        padded = list(pad_both_ends(sent, n))
        test_ngrams = list(ngrams(padded, n))
        try:
            pp = model.perplexity(test_ngrams)
            if np.isfinite(pp):
                perplexities.append(pp)
        except Exception:
            pass
    return np.mean(perplexities) if perplexities else float('inf')

NUM_SENTS = 10
pp_mle = compute_perplexity(model_mle, n, test_sents[:NUM_SENTS])
pp_lap = compute_perplexity(model_lap, n, test_sents[:NUM_SENTS])
pp_kn  = compute_perplexity(model_kn,  n, test_sents[:NUM_SENTS])

print(f'Perplexity on test set (n={n})')
print(f'  MLE       : {pp_mle:.2f}')
print(f'  Laplace   : {pp_lap:.2f}')
print(f'  Kneser-Ney: {pp_kn:.2f}')

Perplexity on test set (n=3)
  MLE       : inf
  Laplace   : 1891.96
  Kneser-Ney: 113.07


**Key insight:** MLE assigns probability 0 to any unseen n-gram in the test set, which makes the perplexity undefined (infinite). Smoothing methods redistribute probability mass to unseen events, making evaluation possible.

## Exercise: Zipf's Law and the Data Sparsity Problem

### Background

**Zipf's Law** states that, in any natural language corpus, the frequency of a word is inversely proportional to its rank in the frequency table:

$$f(r) \propto \frac{1}{r^\alpha}, \quad \alpha \approx 1$$

In log-log space this becomes a straight line with slope $-\alpha$. Your task is to verify this empirically and connect it to what you have observed during the session.

### Unigram frequency distribution

For the experiment, we will use the Brown corpus, which is larger and more varied than Emma.

The Brown Corpus (1961, Brown University) was the first computational corpus of modern 
English. It contains ~1 million tokens spanning 15 genres: news, fiction, government 
documents, academic writing, and more. Its size and lexical variety make it well suited 
for studying the statistical properties of natural language, such as Zipf's Law.

In [15]:
# Load and lowercase the Brown corpus
brown_words = [w.lower() for w in brown.words()]
print(f'Total tokens in Brown corpus: {len(brown_words)}')

Total tokens in Brown corpus: 1161192


In [16]:
# --- YOUR CODE HERE ---
# 1. Count word frequencies using Counter
# 2. Sort by frequency (descending)
# 3. Extract ranks and frequencies as numpy arrays


si provem en lloc de amb el brown amb el emma o noseque hauriem de veure grafics similars. ha fet un grafic en escala lineal i una en logaritmica. la logaritmica era una recta de pendent negativa

In [17]:
# --- YOUR CODE HERE ---
# Plot frequency vs. rank in log-log scale
# Add a reference line with slope -1 to compare with ideal Zipf's Law


In [18]:
# --- YOUR CODE HERE ---
# Fit a line in log-log space using numpy.polyfit to estimate the exponent alpha
# Use only the top 5000 ranks to avoid noise from hapax legomena
